In [1]:
#importamos librerías necesarias
import pandas as pd
import requests
import os
from dotenv import load_dotenv
import tarfile
import gzip
import io
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import time
import re

In [15]:
# cargamos API_KEY
load_dotenv()

API_KEY = os.getenv("AEMET_API_KEY")

if API_KEY is None:
    raise ValueError("No se ha encontrado la API Key. Revisa el archivo .env")
else:
    print("API Key cargada correctamente")

python-dotenv could not parse statement starting at line 2


API Key cargada correctamente


In [17]:
# definimos fechas del proyecto 
fecha_inicio_total = datetime(2021, 1, 1)
fecha_fin_total = datetime(2026, 4, 30)

print("Fecha inicio:", fecha_inicio_total)
print("Fecha fin:", fecha_fin_total)

Fecha inicio: 2021-01-01 00:00:00
Fecha fin: 2026-04-30 00:00:00


In [18]:
# Creamos rango de dos dias para la extracción de datos
rangos_fechas = []

fecha_actual = fecha_inicio_total

while fecha_actual <= fecha_fin_total:
    inicio = fecha_actual
    fin = min(fecha_actual + timedelta(days=1), fecha_fin_total)

    rangos_fechas.append((inicio, fin))

    fecha_actual = fin + timedelta(days=1)

print("Número de rangos creados:", len(rangos_fechas))

for inicio, fin in rangos_fechas[:10]:
    print(inicio.strftime("%Y-%m-%d"), "→", fin.strftime("%Y-%m-%d"))

print("...")

for inicio, fin in rangos_fechas[-10:]:
    print(inicio.strftime("%Y-%m-%d"), "→", fin.strftime("%Y-%m-%d"))

Número de rangos creados: 973
2021-01-01 → 2021-01-02
2021-01-03 → 2021-01-04
2021-01-05 → 2021-01-06
2021-01-07 → 2021-01-08
2021-01-09 → 2021-01-10
2021-01-11 → 2021-01-12
2021-01-13 → 2021-01-14
2021-01-15 → 2021-01-16
2021-01-17 → 2021-01-18
2021-01-19 → 2021-01-20
...
2026-04-11 → 2026-04-12
2026-04-13 → 2026-04-14
2026-04-15 → 2026-04-16
2026-04-17 → 2026-04-18
2026-04-19 → 2026-04-20
2026-04-21 → 2026-04-22
2026-04-23 → 2026-04-24
2026-04-25 → 2026-04-26
2026-04-27 → 2026-04-28
2026-04-29 → 2026-04-30


In [19]:
# funciones auxiliares para leer el XML de AEMET
def limpiar_tag(tag):
    if "}" in tag:
        return tag.split("}", 1)[1]
    return tag


def hijos(elemento, nombre):
    return [hijo for hijo in list(elemento) if limpiar_tag(hijo.tag) == nombre]


def texto(elemento, nombre):
    for hijo in list(elemento):
        if limpiar_tag(hijo.tag) == nombre:
            return hijo.text if hijo.text is not None else ""
    return ""

In [20]:
#funciones para descargar un rango de 2 dias 
def descargar_rango_aemet(fecha_inicio, fecha_fin, api_key, pausa=10, max_intentos=5):
    fecha_ini = fecha_inicio.strftime("%Y-%m-%dT00:00:00UTC")
    fecha_fin_str = fecha_fin.strftime("%Y-%m-%dT23:59:59UTC")

    url = (
        "https://opendata.aemet.es/opendata/api/avisos_cap/archivo/"
        f"fechaini/{fecha_ini}/fechafin/{fecha_fin_str}"
    )

    params = {
        "api_key": API_KEY
    }

    for intento in range(1, max_intentos + 1):
        print(f"Intento {intento}/{max_intentos}: {fecha_ini} → {fecha_fin_str}")

        # Primera llamada: pedir a AEMET la URL temporal
        response = requests.get(url, params=params, timeout=90)

        if response.status_code == 429:
            print("Demasiadas peticiones. Esperando 90 segundos...")
            time.sleep(90)
            continue

        if response.status_code != 200:
            print("Error en primera petición:", response.status_code)
            print(response.text[:500])
            time.sleep(pausa)
            continue

        try:
            respuesta_json = response.json()
        except Exception:
            print("La respuesta no es JSON")
            print(response.text[:500])
            time.sleep(pausa)
            continue

        url_datos = respuesta_json.get("datos")

        if not url_datos:
            print("No se encontró URL de datos")
            print(respuesta_json)
            time.sleep(pausa)
            continue

        time.sleep(pausa)

        # Segunda llamada: descargar el archivo real
        datos_response = requests.get(url_datos)

        if datos_response.status_code == 429:
            print("Demasiadas peticiones en descarga. Esperando 90 segundos...")
            time.sleep(90)
            continue

        if datos_response.status_code == 500:
            print("Error 500 en AEMET. Esperando y reintentando...")
            time.sleep(60)
            continue

        if datos_response.status_code != 200:
            print("Error descargando archivo real:", datos_response.status_code)
            print(datos_response.text[:500])
            time.sleep(pausa)
            continue

        contenido = datos_response.content

        # Comprobación mínima: parece TAR si empieza con nombre de archivo Z_CAP...
        if not contenido.startswith(b"Z_CAP"):
            print("El contenido descargado no parece el archivo esperado.")
            print(contenido[:300])
            time.sleep(pausa)
            continue

        print("Descarga correcta")
        return contenido

    print("No se pudo descargar este rango después de varios intentos.")
    return None

In [21]:
# Función para convertir TAR + GZ + XML en tabla
def convertir_contenido_a_dataframe(contenido):
    filas = []

    try:
        archivo_tar = tarfile.open(fileobj=io.BytesIO(contenido), mode="r:")
    except tarfile.TarError:
        print("No se pudo abrir como TAR")
        return pd.DataFrame()

    for miembro in archivo_tar.getmembers():

        if not miembro.name.endswith(".gz"):
            continue

        archivo_extraido = archivo_tar.extractfile(miembro)

        if archivo_extraido is None:
            continue

        contenido_gz = archivo_extraido.read()

        try:
            contenido_xml = gzip.decompress(contenido_gz)
        except gzip.BadGzipFile:
            continue

        texto_xml = contenido_xml.decode("utf-8", errors="ignore")

        bloques_xml = re.findall(
            r"<\?xml.*?</alert>",
            texto_xml,
            flags=re.DOTALL
        )

        for bloque in bloques_xml:

            try:
                root = ET.fromstring(bloque)
            except ET.ParseError:
                continue

            if limpiar_tag(root.tag) == "alert":
                alertas = [root]
            else:
                alertas = [
                    elem for elem in root.iter()
                    if limpiar_tag(elem.tag) == "alert"
                ]

            for alert in alertas:

                identifier = texto(alert, "identifier")
                sender = texto(alert, "sender")
                sent = texto(alert, "sent")
                status = texto(alert, "status")
                msg_type = texto(alert, "msgType")
                scope = texto(alert, "scope")

                for info in hijos(alert, "info"):

                    language = texto(info, "language")
                    category = texto(info, "category")
                    event = texto(info, "event")
                    urgency = texto(info, "urgency")
                    severity = texto(info, "severity")
                    certainty = texto(info, "certainty")
                    effective = texto(info, "effective")
                    onset = texto(info, "onset")
                    expires = texto(info, "expires")
                    headline = texto(info, "headline")
                    description = texto(info, "description")

                    areas = hijos(info, "area")

                    for area in areas:

                        area_desc = texto(area, "areaDesc")
                        polygon = texto(area, "polygon")
                        circle = texto(area, "circle")

                        geocode_valores = []

                        for geocode in hijos(area, "geocode"):
                            value_name = texto(geocode, "valueName")
                            value = texto(geocode, "value")

                            if value_name or value:
                                geocode_valores.append(f"{value_name}: {value}")

                        geocode_texto = " | ".join(geocode_valores)

                        filas.append({
                            "identifier": identifier,
                            "sender": sender,
                            "sent": sent,
                            "status": status,
                            "msg_type": msg_type,
                            "scope": scope,
                            "language": language,
                            "category": category,
                            "event": event,
                            "urgency": urgency,
                            "severity": severity,
                            "certainty": certainty,
                            "effective": effective,
                            "onset": onset,
                            "expires": expires,
                            "headline": headline,
                            "description": description,
                            "area_desc": area_desc,
                            "polygon": polygon,
                            "circle": circle,
                            "geocode": geocode_texto,
                            "archivo_origen": miembro.name
                        })

    return pd.DataFrame(filas)

In [22]:
# funciones para limpiar y crear nuevas columnas 
def limpiar_dataframe(df):
    if df.empty:
        return df

    df_limpio = df.drop_duplicates(
        subset=[
            "identifier",
            "sent",
            "onset",
            "expires",
            "event",
            "severity",
            "area_desc",
            "headline"
        ]
    ).copy()

    # Convertimos fechas usando utc=True para evitar errores con zonas horarias mixtas
    df_limpio["onset"] = pd.to_datetime(df_limpio["onset"], errors="coerce", utc=True)
    df_limpio["expires"] = pd.to_datetime(df_limpio["expires"], errors="coerce", utc=True)
    df_limpio["sent"] = pd.to_datetime(df_limpio["sent"], errors="coerce", utc=True)

    # Creamos año y mes
    df_limpio["año"] = df_limpio["onset"].dt.year
    df_limpio["mes_num"] = df_limpio["onset"].dt.month

    nombres_meses = {
        1: "Enero",
        2: "Febrero",
        3: "Marzo",
        4: "Abril",
        5: "Mayo",
        6: "Junio",
        7: "Julio",
        8: "Agosto",
        9: "Septiembre",
        10: "Octubre",
        11: "Noviembre",
        12: "Diciembre"
    }

    df_limpio["mes"] = df_limpio["mes_num"].map(nombres_meses)

    df_limpio["estacion"] = df_limpio["mes_num"].map({
        12: "Invierno",
        1: "Invierno",
        2: "Invierno",
        3: "Primavera",
        4: "Primavera",
        5: "Primavera",
        6: "Verano",
        7: "Verano",
        8: "Verano",
        9: "Otoño",
        10: "Otoño",
        11: "Otoño"
    })

    df_limpio["duracion_horas"] = (
        df_limpio["expires"] - df_limpio["onset"]
    ).dt.total_seconds() / 3600

    df_limpio["riesgo_numerico"] = df_limpio["severity"].map({
        "Minor": 0,
        "Moderate": 1,
        "Severe": 2,
        "Extreme": 3
    }).fillna(0)

    return df_limpio

In [9]:
# probamos con un solo rango antes que todo 
contenido_prueba = descargar_rango_aemet(
    datetime(2021, 1, 7),
    datetime(2021, 1, 8),
    API_KEY,
    pausa=12
)

df_prueba = convertir_contenido_a_dataframe(contenido_prueba)

df_prueba_limpio = limpiar_dataframe(df_prueba)

print("Filas prueba:", len(df_prueba_limpio))

df_prueba_limpio.head()

Intento 1/5: 2021-01-07T00:00:00UTC → 2021-01-08T23:59:59UTC
Descarga correcta
Filas prueba: 12604


,identifier,sender,sent,status,msg_type,scope,language,category,event,urgency,...,polygon,circle,geocode,archivo_origen,año,mes_num,mes,estacion,duracion_horas,riesgo_numerico
0,2.49.0.0.724.0.ES.20210107043640.750102BTTI070...,http://www.aemet.es,2021-01-07 04:36:40+00:00,Actual,Alert,Public,es-ES,Met,Aviso de temperaturas mínimas de nivel amarillo,Immediate,...,"42.94,-2.98 42.94,-2.93 42.91,-2.86 42.92,-2.7...",,AEMET-Meteoalerta zona: 750102,Z_CAP_C_LEMM_20210107043640_AFAE.gz,2021,1,Enero,Invierno,3.999722,1
1,2.49.0.0.724.0.ES.20210107043640.750102BTTI070...,http://www.aemet.es,2021-01-07 04:36:40+00:00,Actual,Alert,Public,en-GB,Met,Moderate low-temperature warning,Immediate,...,"42.94,-2.98 42.94,-2.93 42.91,-2.86 42.92,-2.7...",,AEMET-Meteoalerta zona: 750102,Z_CAP_C_LEMM_20210107043640_AFAE.gz,2021,1,Enero,Invierno,3.999722,1
2,2.49.0.0.724.0.ES.20210106225001.624401NENV092...,http://www.aemet.es,2021-01-06 22:50:01+00:00,Actual,Alert,Public,es-ES,Met,Aviso de nevadas de nivel naranja,Future,...,"40.94,-1.62 40.94,-1.56 40.96,-1.5 40.96,-1.47...",,AEMET-Meteoalerta zona: 624401,Z_CAP_C_LEMM_20210107043640_AFAE.gz,2021,1,Enero,Invierno,23.999722,2
3,2.49.0.0.724.0.ES.20210106225001.624401NENV092...,http://www.aemet.es,2021-01-06 22:50:01+00:00,Actual,Alert,Public,en-GB,Met,Severe snow warning,Future,...,"40.94,-1.62 40.94,-1.56 40.96,-1.5 40.96,-1.47...",,AEMET-Meteoalerta zona: 624401,Z_CAP_C_LEMM_20210107043640_AFAE.gz,2021,1,Enero,Invierno,23.999722,2
4,2.49.0.0.724.0.ES.20210106225001.624402NENV092...,http://www.aemet.es,2021-01-06 22:50:01+00:00,Actual,Alert,Public,es-ES,Met,Aviso de nevadas de nivel naranja,Future,...,"39.97,-1.14 40.01,-1.16 40.04,-1.08 40.06,-1.0...",,AEMET-Meteoalerta zona: 624402,Z_CAP_C_LEMM_20210107043640_AFAE.gz,2021,1,Enero,Invierno,23.999722,2


In [6]:
# Crear carpeta por meses 
import os

carpeta_salida = "avisos_aemet_por_mes"

os.makedirs(carpeta_salida, exist_ok=True)

print("Carpeta lista:", carpeta_salida)

Carpeta lista: avisos_aemet_por_mes


In [20]:
# creamos la lista de meses desde enero de 2021 hasta abril de 2026
meses_a_procesar = []

for anio in range(2021, 2027):
    for mes in range(1, 13):

        if anio == 2026 and mes > 4:
            continue

        meses_a_procesar.append((anio, mes))

print("Número de meses:", len(meses_a_procesar))
print(meses_a_procesar[:5])
print(meses_a_procesar[-5:])

Número de meses: 64
[(2021, 1), (2021, 2), (2021, 3), (2021, 4), (2021, 5)]
[(2025, 12), (2026, 1), (2026, 2), (2026, 3), (2026, 4)]


In [24]:
meses_a_procesar = []

for anio in range(2023, 2027):
    for mes in range(1, 13):

        if anio == 2023 and mes < 6:
            continue

        if anio == 2026 and mes > 4:
            continue

        meses_a_procesar.append((anio, mes))

print("Meses pendientes:", len(meses_a_procesar))
print(meses_a_procesar[:5])
print(meses_a_procesar[-5:])

Meses pendientes: 35
[(2023, 6), (2023, 7), (2023, 8), (2023, 9), (2023, 10)]
[(2025, 12), (2026, 1), (2026, 2), (2026, 3), (2026, 4)]


In [13]:
# descargar y guardar un CSV por cada mes
archivos_generados = []

for anio, mes in meses_a_procesar:

    print("=" * 80)
    print(f"PROCESANDO {anio}-{mes:02d}")
    print("=" * 80)

    # Primer y último día del mes
    fecha_inicio_mes = datetime(anio, mes, 1)

    if mes == 12:
        fecha_inicio_siguiente_mes = datetime(anio + 1, 1, 1)
    else:
        fecha_inicio_siguiente_mes = datetime(anio, mes + 1, 1)

    fecha_fin_mes = fecha_inicio_siguiente_mes - timedelta(days=1)

    # Dividir el mes en bloques de máximo 2 días
    rangos_mes = []
    fecha_actual = fecha_inicio_mes

    while fecha_actual <= fecha_fin_mes:
        inicio = fecha_actual
        fin = min(fecha_actual + timedelta(days=1), fecha_fin_mes)

        rangos_mes.append((inicio, fin))

        fecha_actual = fin + timedelta(days=1)

    print("Rangos de 2 días en este mes:", len(rangos_mes))

    dfs_mes = []

    for inicio, fin in rangos_mes:

        contenido = descargar_rango_aemet(
            inicio,
            fin,
            API_KEY,
            pausa=8,
            max_intentos=5
        )

        if contenido is None:
            print("No se pudo descargar este rango. Se pasa al siguiente.")
            continue

        df_rango = convertir_contenido_a_dataframe(contenido)

        if df_rango.empty:
            print("No se encontraron datos en este rango.")
            continue

        df_rango_limpio = limpiar_dataframe(df_rango)

        if df_rango_limpio.empty:
            print("El rango quedó vacío después de limpiar.")
            continue

        dfs_mes.append(df_rango_limpio)

        print("Filas limpias del rango:", len(df_rango_limpio))

    if len(dfs_mes) == 0:
        print(f"No se obtuvieron datos para {anio}-{mes:02d}")
        continue

    # Unir todos los rangos del mes
    df_mes = pd.concat(dfs_mes, ignore_index=True)

    # Quitar duplicados dentro del mes
    df_mes_limpio = df_mes.drop_duplicates(
        subset=[
            "identifier",
            "sent",
            "onset",
            "expires",
            "event",
            "severity",
            "area_desc",
            "headline"
        ]
    ).copy()

    print("Filas del mes antes de limpieza final:", len(df_mes))
    print("Filas del mes después de limpieza final:", len(df_mes_limpio))

    nombre_archivo = f"{carpeta_salida}/avisos_aemet_{anio}_{mes:02d}.csv"

    df_mes_limpio.to_csv(
        nombre_archivo,
        index=False,
        encoding="utf-8-sig"
    )

    archivos_generados.append(nombre_archivo)

    print("Archivo mensual guardado:", nombre_archivo)

NameError: name 'meses_a_procesar' is not defined

In [14]:
# comprobar archivos mensuales generados
print("Número de archivos mensuales generados:", len(archivos_generados))

archivos_generados[:10]

Número de archivos mensuales generados: 0


[]

In [18]:
import pandas as pd
import glob
import os

In [19]:
archivos = sorted(glob.glob("avisos_aemet_por_mes/*.csv"))

print("Archivos encontrados:", len(archivos))
print("Primeros archivos:")
print(archivos[:5])
print("Últimos archivos:")
print(archivos[-5:])

Archivos encontrados: 64
Primeros archivos:
['avisos_aemet_por_mes\\avisos_aemet_2021_01.csv', 'avisos_aemet_por_mes\\avisos_aemet_2021_02.csv', 'avisos_aemet_por_mes\\avisos_aemet_2021_03.csv', 'avisos_aemet_por_mes\\avisos_aemet_2021_04.csv', 'avisos_aemet_por_mes\\avisos_aemet_2021_05.csv']
Últimos archivos:
['avisos_aemet_por_mes\\avisos_aemet_2025_12.csv', 'avisos_aemet_por_mes\\avisos_aemet_2026_01.csv', 'avisos_aemet_por_mes\\avisos_aemet_2026_02.csv', 'avisos_aemet_por_mes\\avisos_aemet_2026_03.csv', 'avisos_aemet_por_mes\\avisos_aemet_2026_04.csv']


In [20]:
archivos = sorted(
    glob.glob(
        r"C:\Users\Marta\Desktop\Adalab\proyectos\proyecto-modulo-4\proyecto-da-promo-63-modulo-4-team2-proyectofinal\avisos_aemet_por_mes\*.csv"
    )
)

print("Archivos encontrados:", len(archivos))
print(archivos[:5])
print(archivos[-5:])

Archivos encontrados: 64
['C:\\Users\\Marta\\Desktop\\Adalab\\proyectos\\proyecto-modulo-4\\proyecto-da-promo-63-modulo-4-team2-proyectofinal\\avisos_aemet_por_mes\\avisos_aemet_2021_01.csv', 'C:\\Users\\Marta\\Desktop\\Adalab\\proyectos\\proyecto-modulo-4\\proyecto-da-promo-63-modulo-4-team2-proyectofinal\\avisos_aemet_por_mes\\avisos_aemet_2021_02.csv', 'C:\\Users\\Marta\\Desktop\\Adalab\\proyectos\\proyecto-modulo-4\\proyecto-da-promo-63-modulo-4-team2-proyectofinal\\avisos_aemet_por_mes\\avisos_aemet_2021_03.csv', 'C:\\Users\\Marta\\Desktop\\Adalab\\proyectos\\proyecto-modulo-4\\proyecto-da-promo-63-modulo-4-team2-proyectofinal\\avisos_aemet_por_mes\\avisos_aemet_2021_04.csv', 'C:\\Users\\Marta\\Desktop\\Adalab\\proyectos\\proyecto-modulo-4\\proyecto-da-promo-63-modulo-4-team2-proyectofinal\\avisos_aemet_por_mes\\avisos_aemet_2021_05.csv']
['C:\\Users\\Marta\\Desktop\\Adalab\\proyectos\\proyecto-modulo-4\\proyecto-da-promo-63-modulo-4-team2-proyectofinal\\avisos_aemet_por_mes\\avis

In [21]:
columnas_finales = [
    "onset",
    "expires",
    "año",
    "mes",
    "mes_num",
    "estacion",
    "event",
    "severity",
    "riesgo_numerico",
    "duracion_horas",
    "area_desc",
    "descripcion",
    "headline",
    "description",
]

columnas_duplicados = ["onset", "expires", "event", "severity", "area_desc"]

In [22]:
archivo_salida = "avisos_aemet_2021_2026_descripcion.csv"

if os.path.exists(archivo_salida):
    os.remove(archivo_salida)

primera_vez = True
total_filas_guardadas = 0

niveles_seleccionados = ["Severe", "Extreme"]

for archivo in archivos:
    print("=" * 80)
    print("Procesando:", archivo)
    print("=" * 80)

    try:
        lector = pd.read_csv(
            archivo,
            chunksize=50000,
            encoding="utf-8-sig",
            low_memory=False,
            on_bad_lines="skip",
        )

        for i, chunk in enumerate(lector):
            print(f"  Bloque {i + 1}")

            if "severity" not in chunk.columns:
                print("  Este bloque no tiene columna severity")
                continue

            # Filtrar solo Moderate, Severe y Extreme
            chunk = chunk[chunk["severity"].isin(niveles_seleccionados)]

            if chunk.empty:
                print("  No hay filas Moderate, Severe o Extreme en este bloque")
                continue

            columnas_disponibles = [
                col for col in columnas_finales if col in chunk.columns
            ]

            chunk = chunk[columnas_disponibles]

            columnas_dup_disponibles = [
                col for col in columnas_duplicados if col in chunk.columns
            ]

            if len(columnas_dup_disponibles) > 0:
                chunk = chunk.drop_duplicates(subset=columnas_dup_disponibles)

            chunk.to_csv(
                archivo_salida,
                mode="w" if primera_vez else "a",
                header=primera_vez,
                index=False,
                encoding="utf-8-sig",
            )

            primera_vez = False
            total_filas_guardadas += len(chunk)

            print("  Filas añadidas:", len(chunk))
            print("  Total acumulado:", total_filas_guardadas)

    except Exception as e:
        print("ERROR con archivo:")
        print(archivo)
        print(e)
        continue

print("Proceso terminado")
print("Archivo creado:", archivo_salida)
print("Filas aproximadas guardadas:", total_filas_guardadas)

Procesando: C:\Users\Marta\Desktop\Adalab\proyectos\proyecto-modulo-4\proyecto-da-promo-63-modulo-4-team2-proyectofinal\avisos_aemet_por_mes\avisos_aemet_2021_01.csv
  Bloque 1
  Filas añadidas: 1332
  Total acumulado: 1332
  Bloque 2
  Filas añadidas: 838
  Total acumulado: 2170
  Bloque 3
  Filas añadidas: 302
  Total acumulado: 2472
Procesando: C:\Users\Marta\Desktop\Adalab\proyectos\proyecto-modulo-4\proyecto-da-promo-63-modulo-4-team2-proyectofinal\avisos_aemet_por_mes\avisos_aemet_2021_02.csv
  Bloque 1
  Filas añadidas: 268
  Total acumulado: 2740
  Bloque 2
  Filas añadidas: 174
  Total acumulado: 2914
Procesando: C:\Users\Marta\Desktop\Adalab\proyectos\proyecto-modulo-4\proyecto-da-promo-63-modulo-4-team2-proyectofinal\avisos_aemet_por_mes\avisos_aemet_2021_03.csv
  Bloque 1
  Filas añadidas: 114
  Total acumulado: 3028
  Bloque 2
  Filas añadidas: 74
  Total acumulado: 3102
  Bloque 3
  No hay filas Moderate, Severe o Extreme en este bloque
Procesando: C:\Users\Marta\Desktop\

In [23]:
archivo = "avisos_aemet_2021_2026_descripcion.csv"

total_filas = 0
columnas = None
conteo_severity = {}

for chunk in pd.read_csv(
    archivo, chunksize=50000, encoding="utf-8-sig", low_memory=False
):
    total_filas += len(chunk)

    if columnas is None:
        columnas = chunk.columns.tolist()

    conteo = chunk["severity"].value_counts()

    for nivel, cantidad in conteo.items():
        conteo_severity[nivel] = conteo_severity.get(nivel, 0) + cantidad

print("Filas:", total_filas)
print("Columnas:", len(columnas))
print("Nombres de columnas:")
print(columnas)

print("Conteo severity:")
print(conteo_severity)

Filas: 44666
Columnas: 13
Nombres de columnas:
['onset', 'expires', 'año', 'mes', 'mes_num', 'estacion', 'event', 'severity', 'riesgo_numerico', 'duracion_horas', 'area_desc', 'headline', 'description']
Conteo severity:
{'Severe': 43018, 'Extreme': 1648}


In [24]:
import os

archivo = "avisos_aemet_2021_2026_descripcion.csv"

tamaño_bytes = os.path.getsize(archivo)
tamaño_mb = tamaño_bytes / (1024 * 1024)

print("Tamaño en bytes:", tamaño_bytes)
print("Tamaño en MB:", round(tamaño_mb, 2))

Tamaño en bytes: 12910228
Tamaño en MB: 12.31


In [29]:
# definimos las columnas
columnas_utiles = [
    "identifier",
    "sent",
    "onset",
    "expires",
    "año",
    "mes",
    "estacion",
    "event",
    "severity",
    "riesgo_numerico",
    "duracion_horas",
    "area_desc",
    "geocode",
]


In [30]:
# columnas duplicadas
columnas_duplicados = [
    "identifier",
    "sent",
    "onset",
    "expires",
    "event",
    "severity",
    "area_desc",
]

In [31]:
archivo_salida = "avisos_aemet_2021_2026_super_ligero_tableau.csv"

if os.path.exists(archivo_salida):
    os.remove(archivo_salida)

primera_vez = True
total_filas_guardadas = 0

for archivo in archivos:
    print("=" * 80)
    print("Procesando:", archivo)
    print("=" * 80)

    try:
        lector = pd.read_csv(
            archivo,
            chunksize=50000,
            encoding="utf-8-sig",
            low_memory=False,
            on_bad_lines="skip",
        )

        for i, chunk in enumerate(lector):
            print(f"  Bloque {i + 1}")

            columnas_disponibles = [
                col for col in columnas_super_ligeras if col in chunk.columns
            ]

            chunk = chunk[columnas_disponibles]

            columnas_dup_disponibles = [
                col for col in columnas_duplicados if col in chunk.columns
            ]

            if len(columnas_dup_disponibles) > 0:
                chunk = chunk.drop_duplicates(subset=columnas_dup_disponibles)

            chunk.to_csv(
                archivo_salida,
                mode="w" if primera_vez else "a",
                header=primera_vez,
                index=False,
                encoding="utf-8-sig",
            )

            primera_vez = False
            total_filas_guardadas += len(chunk)

            print("  Filas añadidas:", len(chunk))
            print("  Total acumulado:", total_filas_guardadas)

    except Exception as e:
        print("ERROR con archivo:")
        print(archivo)
        print(e)
        continue

print("Proceso terminado")
print("Archivo creado:", archivo_salida)
print("Filas aproximadas guardadas:", total_filas_guardadas)

Procesando: C:\Users\Marta\Desktop\Adalab\proyectos\proyecto-modulo-4\proyecto-da-promo-63-modulo-4-team2-proyectofinal\avisos_aemet_por_mes\avisos_aemet_2021_01.csv
  Bloque 1
ERROR con archivo:
C:\Users\Marta\Desktop\Adalab\proyectos\proyecto-modulo-4\proyecto-da-promo-63-modulo-4-team2-proyectofinal\avisos_aemet_por_mes\avisos_aemet_2021_01.csv
name 'columnas_super_ligeras' is not defined
Procesando: C:\Users\Marta\Desktop\Adalab\proyectos\proyecto-modulo-4\proyecto-da-promo-63-modulo-4-team2-proyectofinal\avisos_aemet_por_mes\avisos_aemet_2021_02.csv
  Bloque 1
ERROR con archivo:
C:\Users\Marta\Desktop\Adalab\proyectos\proyecto-modulo-4\proyecto-da-promo-63-modulo-4-team2-proyectofinal\avisos_aemet_por_mes\avisos_aemet_2021_02.csv
name 'columnas_super_ligeras' is not defined
Procesando: C:\Users\Marta\Desktop\Adalab\proyectos\proyecto-modulo-4\proyecto-da-promo-63-modulo-4-team2-proyectofinal\avisos_aemet_por_mes\avisos_aemet_2021_03.csv
  Bloque 1
ERROR con archivo:
C:\Users\Marta

In [24]:
archivo = "avisos_aemet_2021_2026_super_ligero_tableau.csv"

total_filas = 0
columnas = None

for chunk in pd.read_csv(
    archivo, chunksize=50000, encoding="utf-8-sig", low_memory=False
):
    total_filas += len(chunk)

    if columnas is None:
        columnas = chunk.columns.tolist()

print("Filas:", total_filas)
print("Columnas:", len(columnas))
print("Nombres de columnas:")
print(columnas)

FileNotFoundError: [Errno 2] No such file or directory: 'avisos_aemet_2021_2026_super_ligero_tableau.csv'

In [9]:
import os

print(os.getcwd())

c:\Users\Marta\Desktop\Adalab\proyectos\proyecto-modulo-4\proyecto-da-promo-63-modulo-4-team2-proyectofinal


In [10]:
import glob

csvs = glob.glob("*.csv")

print("CSV encontrados:", len(csvs))
csvs

CSV encontrados: 4


['datos_dana_emergencias.csv',
 'datos_dana_unificados.csv',
 'datos_filomena_emergencias.csv',
 'datos_filomena_unificados.csv']

In [11]:
csvs_todos = glob.glob("**/*.csv", recursive=True)

print("CSV encontrados en subcarpetas:", len(csvs_todos))
csvs_todos[:50]

CSV encontrados en subcarpetas: 68


['datos_dana_emergencias.csv',
 'datos_dana_unificados.csv',
 'datos_filomena_emergencias.csv',
 'datos_filomena_unificados.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2021_01.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2021_02.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2021_03.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2021_04.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2021_05.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2021_06.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2021_07.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2021_08.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2021_09.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2021_10.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2021_11.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2021_12.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2022_01.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2022_02.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2022_03.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2022_04.csv',
 'avisos_aemet_por_mes\\avisos_aemet_2022_05.csv',
 'a

In [1]:
import zipfile

archivo_csv = "avisos_aemet_2021_2026_moderate_severe_extreme.csv"
archivo_zip = "avisos_aemet_2021_2026_moderate_severe_extreme.zip"

with zipfile.ZipFile(archivo_zip, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(archivo_csv)

print("ZIP creado:", archivo_zip)

ZIP creado: avisos_aemet_2021_2026_moderate_severe_extreme.zip
